# 01 · Data Loading and the Programmatic Oracle
**Grounded Conversational Chart Accessibility** — Section 1 of 6

**What this notebook does, in plain language**

1. Downloads the ChartQA test set (chart data tables + question/answer pairs).
2. Cleans each table so `"36%"`, `"$1,200"` or `"457*"` become real numbers.
3. Labels every question with a **type** (lookup, max, min, sum, avg, diff, compare, trend) and every chart with a **chart type** (bar / line / pie).
4. Runs the **oracle**: plain pandas code that computes the true answer from the table. No LLM is involved.
5. **Checks the oracle against ChartQA's human gold answers** and keeps only questions it gets right. This is our trustworthy evaluation set.
6. Draws a balanced sample for the LLM experiments in Section 2.

The functions live in `src/` (so later notebooks can reuse them); this notebook explains and runs them.

## 1.1 Setup

In [1]:
# First time only:  !pip install -r ../requirements.txt
import sys, logging
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))     # lets us `import src` from the notebooks/ folder

import pandas as pd
from src import config
from src.data_utils import download_chartqa, load_qa, load_table, infer_chart_type
from src.oracle import classify_question, oracle, matches_gold, mentioned

config.make_dirs()
config.set_seeds()                               # SEED = 42, for reproducibility

# Every skipped table/question is written here, with the reason
logging.basicConfig(filename=config.LOG_DIR / "01_skipped.log", filemode="w",
                    level=logging.INFO, format="%(levelname)s | %(message)s", force=True)
log = logging.getLogger("pipeline")
print("Project root:", config.ROOT)

Project root: D:\clg\IV\chart-hallucination


## 1.2 Download ChartQA and load the questions

Only the **test** split's tables and Q&A files are downloaded (a few MB, no images).

ChartQA has two kinds of questions:
- **human**: written by people; often mention colours ("the blue bar") that a table can't resolve.
- **augmented**: machine-generated; usually name the exact entity ("What was X in 2019?").

In [2]:
download_chartqa()
qa = load_qa()
print(qa["source"].value_counts().to_string())
qa.head()

Downloaded ChartQA test split.
source
human        1250
augmented    1250


,qid,imgname,query,label,source,table_file
0,0,41699051005347.png,How many food item is shown in the bar graph?,14,human,41699051005347.csv
1,1,41699051005347.png,What is the difference in value between Lamb a...,0.57,human,41699051005347.csv
2,2,41810321001157.png,How many bars are shown in the chart?,3,human,41810321001157.csv
3,3,41810321001157.png,Is the sum value of Madagascar more then Fiji?,No,human,41810321001157.csv
4,4,8127.png,What's the value of the lowest bar?,23,human,8127.csv


## 1.3 Load and clean the tables

`load_table` (in `src/data_utils.py`) applies one convention: the **first column holds labels** (categories or years), every other column is a **data series**. `to_number` cleans each cell:

| raw cell | becomes |
|---|---|
| `36%` | 36.0 |
| `$1,200.5` | 1200.5 |
| `457*` | 457.0 |
| `-` or empty | NaN |

Unreadable tables are logged and skipped instead of crashing the notebook.

In [3]:
TABLES = {f: load_table(f) for f in qa["table_file"].unique()}
print(f"Usable tables: {sum(t is not None for t in TABLES.values())} / {len(TABLES)}")
TABLES["two_col_100060.csv"].head()        # an example: years -> exports

Usable tables: 1508 / 1509


,Exports in million U.S. dollars
Characteristic,
2010,23.68
2009,24.84
2008,31.02


## 1.4 Chart type (a heuristic)

**Limitation to state in the report:** ChartQA's GitHub tables don't record whether each chart was a bar, line or pie. We infer it from the table's shape:

- **line**: labels are years/dates, with at least 5 points,
- **pie**: one series, ≤ 8 slices, values add up to about 100,
- **bar**: everything else.

In [4]:
qa["chart_type"] = qa["table_file"].map({f: infer_chart_type(t) for f, t in TABLES.items()})
qa["chart_type"].value_counts(dropna=False)

chart_type
bar     1414
line     912
pie      172
NaN        2
Name: count, dtype: int64

## 1.5 Question type

`classify_question` gives each question one type using keyword rules (see `src/oracle.py`).

| type | example | answer |
|---|---|---|
| `lookup` | What was X's revenue in 2019? | number |
| `max` / `min` | Which country has the highest share? | label or number |
| `sum` / `avg` | What is the average of A, B and C? | number |
| `diff` | Difference between the highest and lowest value? | number |
| `compare` | Is A greater than B? / Ratio of A to B? | Yes/No or number |
| `trend` | Which year saw the biggest increase? | year |

Questions are **skipped** (`skip_*`) when the table can't answer them safely:
- `skip_color`: mentions colours ("the blue bar"),
- `skip_count`: counts chart elements ("how many bars"),
- `skip_complex`: multi-step ("is the sum of the two smallest greater than…"),
- `skip_yesno`: Yes/No questions that aren't a simple A-vs-B comparison.

In [5]:
qa["qtype"] = qa["query"].map(classify_question)
pd.crosstab(qa["qtype"], qa["source"], margins=True)

source,augmented,human,All
qtype,,,
avg,3,36,39
compare,6,35,41
diff,2,96,98
lookup,1052,323,1375
max,90,137,227
min,8,42,50
skip_color,9,241,250
skip_complex,67,193,260
skip_count,2,82,84


## 1.6 How the oracle finds entities

Before computing anything, the oracle must work out **which rows/columns** a question mentions (`mentioned()` in `src/oracle.py`):
- **Years** must match exactly; fuzzy matching would treat `2019` and `2010` as similar.
- **Text labels** use RapidFuzz `partial_ratio ≥ 90`, which forgives small spelling differences.

In [6]:
t = TABLES["two_col_100060.csv"]
print(mentioned(t.index, "What were exports worth in 2009?"))      # -> ['2009']

['2009']


## 1.7 Run the oracle and check it against the gold answers

For each question the oracle returns either an answer (with its kind: `numeric`, `categorical`, `yesno`) or **refuses**, giving a reason. It refuses whenever anything is ambiguous, because a wrong "truth" would corrupt every later result.

We compare each answer with ChartQA's human gold label using the standard **relaxed accuracy** rule: numbers within **5%**, labels equal after cleaning.

In [7]:
records = []
for r in qa.itertuples():
    result, reason = oracle(r.query, r.qtype, TABLES.get(r.table_file))
    if result is None:
        log.info(f"oracle_skip | {reason} | qid={r.qid} | {r.query}")
        records.append({"qid": r.qid, "oracle_answer": None, "answer_kind": None,
                        "oracle_status": "skipped", "skip_reason": reason})
        continue
    ok = matches_gold(result, r.label)
    if not ok:
        log.info(f"oracle_mismatch | qid={r.qid} | {r.query} | oracle={result['answer']} | gold={r.label}")
    records.append({"qid": r.qid, "oracle_answer": result["answer"],
                    "answer_kind": result["answer_kind"],
                    "oracle_status": "correct" if ok else "mismatch", "skip_reason": None})

qa = qa.merge(pd.DataFrame(records), on="qid")

answered = qa[qa["oracle_status"] != "skipped"]
oracle_report = (answered.groupby("qtype")["oracle_status"]
                 .agg(answered="size", correct=lambda s: (s == "correct").sum()))
oracle_report["precision_%"] = (100 * oracle_report["correct"] / oracle_report["answered"]).round(1)
oracle_report.to_csv(config.TABLE_OUT / "oracle_precision.csv")
oracle_report.sort_values("answered", ascending=False)

,answered,correct,precision_%
qtype,,,
lookup,824,800,97.1
max,146,121,82.9
diff,50,48,96.0
min,25,22,88.0
compare,17,14,82.4
avg,13,12,92.3
sum,13,12,92.3
trend,6,5,83.3


**Reading this table:** `answered` = questions the oracle was confident enough to try; `correct` = how many agree with the human gold label; `precision_%` = how often the oracle is right when it answers.

This is an honest limitation to report: on unseen questions the oracle's rules are roughly this accurate. For our **evaluation**, though, we keep only the `correct` rows, so every "true answer" is confirmed twice, by our code *and* by a human label.

Why questions were skipped:

In [8]:
qa["skip_reason"].value_counts()

skip_reason
lookup_ambiguous              549
skip_complex                  260
skip_color                    249
series_ambiguous              133
skip_count                     84
maxmin_names_entity            25
diff_needs_two_entities        20
skip_yesno                     19
avg_ambiguous                  17
compare_needs_two_entities     16
sum_ambiguous                  14
trend_vague                    11
tie                             4
missing_value                   2
no_table                        2
trend_not_time_series           1
Name: count, dtype: int64

## 1.8 Build the evaluation sample

From the oracle-**correct** questions we draw up to `PER_TYPE_CAP` per type, so common types (lookup) don't swamp rare ones, then top up to `N_TARGET` at random. Rare types simply contribute all they have. Report those small counts honestly.

In [9]:
pool = qa[(qa["oracle_status"] == "correct") & qa["chart_type"].notna()]

parts = [g.sample(min(len(g), config.PER_TYPE_CAP), random_state=config.SEED)
         for _, g in pool.groupby("qtype")]
sample = pd.concat(parts)
need = config.N_TARGET - len(sample)
if need > 0:
    leftovers = pool.drop(sample.index)
    sample = pd.concat([sample, leftovers.sample(min(need, len(leftovers)), random_state=config.SEED)])
sample = sample.sample(frac=1, random_state=config.SEED).reset_index(drop=True)   # shuffle

keep = ["qid", "source", "table_file", "chart_type", "qtype", "query",
        "label", "oracle_answer", "answer_kind"]
sample[keep].to_csv(config.OUTPUT_DIR / "eval_sample.csv", index=False)

print(f"Validated pool: {len(pool)} | sample saved: {len(sample)} rows -> outputs/eval_sample.csv")
pd.crosstab(sample["qtype"], sample["chart_type"], margins=True)

Validated pool: 1034 | sample saved: 200 rows -> outputs/eval_sample.csv


chart_type,bar,line,pie,All
qtype,,,,
avg,7,1,4,12
compare,9,0,5,14
diff,26,15,4,45
lookup,17,28,0,45
max,28,8,9,45
min,12,7,3,22
sum,3,4,5,12
trend,0,5,0,5
All,102,68,30,200


## ✅ Checkpoint before Section 2

1. `outputs/eval_sample.csv` exists with the expected number of rows.
2. Open `outputs/logs/01_skipped.log` and skim ~10 skip/mismatch reasons. Do you agree with them?
3. **Spot-check 5 rows by hand** using the cell below: open the CSV, work out the answer yourself, compare with `oracle_answer`.

In [10]:
row = sample.iloc[0]                       # change 0 -> 1, 2, 3 ... to check others
print(row["query"], "\nGold:", row["label"], "| Oracle:", row["oracle_answer"])
TABLES[row["table_file"]]

How many liters of non-alcoholic beverages were consumed in the UK in 2018? 
Gold: 13779.1 | Oracle: 13779.1


,Total consumption in million liters
Characteristic,
2018,13779.1
2017,13260.9
2016,13294.5
2015,13131.8
2014,13149.3
2013,13213.1
